# Lesson 12: Semantic Segmentation & U-Net Architecture

In this notebook, you will:
1. Build a self-contained **PyTorch U-Net** with encoder–decoder skip connections.
2. Train it on synthetic shape segmentation masks (or Oxford-IIIT Pet trimaps) on Colab Free Tier T4 GPU.

In [ ]:
import torch
import torch.nn as nn
import numpy as np

class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1), nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)

class MiniUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = DoubleConv(1, 16)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(16, 32)
        self.up = nn.ConvTranspose2d(32, 16, 2, stride=2)
        self.dec1 = DoubleConv(32, 16)
        self.head = nn.Conv2d(16, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        b = self.bottleneck(self.pool(e1))
        d1 = self.dec1(torch.cat([self.up(b), e1], dim=1))
        return self.head(d1)

# Generate synthetic circle images and binary segmentation masks
def make_batch(bs=16, sz=32):
    imgs, masks = np.zeros((bs, 1, sz, sz), np.float32), np.zeros((bs, 1, sz, sz), np.float32)
    yy, xx = np.ogrid[:sz, :sz]
    for i in range(bs):
        cx, cy, r = np.random.randint(8, 24), np.random.randint(8, 24), np.random.randint(4, 8)
        m = ((xx - cx)**2 + (yy - cy)**2) <= r**2
        masks[i, 0] = m.astype(np.float32)
        imgs[i, 0] = m.astype(np.float32) * 0.8 + np.random.rand(sz, sz).astype(np.float32) * 0.2
    return torch.tensor(imgs), torch.tensor(masks)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
unet = MiniUNet().to(device)
opt = torch.optim.Adam(unet.parameters(), lr=1e-3)
bce = nn.BCEWithLogitsLoss()

for step in range(60):
    xb, yb = make_batch()
    xb, yb = xb.to(device), yb.to(device)
    opt.zero_grad()
    loss = bce(unet(xb), yb)
    loss.backward()
    opt.step()
    if (step + 1) % 20 == 0:
        print(f"Step {step+1:2d} | Pixel BCE Loss: {loss.item():.4f}")

## Part 2: Evaluating Predicted Masks with IoU & Dice Score

In [ ]:
unet.eval()
xb, yb = make_batch(bs=4)
with torch.no_grad():
    preds = (torch.sigmoid(unet(xb.to(device))) > 0.5).float().cpu()
inter = (preds * yb).sum().item()
union = ((preds + yb) > 0).float().sum().item()
print(f"Validation Batch IoU: {inter / max(union, 1.0):.4f}")